# ETHUSDT Quantitative Research Platform — Notebook 02
## Validation, Robustness, Ablation & Resumable Optuna Studies

This notebook provides systematic statistical validation, robustness stress testing, and hyperparameter optimization.

### Supported Modes:
- `WALK_FORWARD`: Multi-fold anchored or rolling chronological out-of-sample evaluation.
- `ROBUSTNESS`: Adverse fee/slippage stress testing, trade profit concentration, and Monte Carlo bootstrap.
- `ABLATION`: Systematic strategy component removal to compute true marginal value.
- `OPTUNA`: Resumable multi-objective hyperparameter optimization.

In [ ]:
# =============================================================================
# 1. VALIDATION CONFIGURATION & PARAMETERS
# =============================================================================
MODE = "WALK_FORWARD"  # Options: 'WALK_FORWARD', 'ROBUSTNESS', 'ABLATION', 'OPTUNA'

SYMBOL = "ETHUSDT"
TIMEFRAME = "15m"

# Target Strategy for Validation
STRATEGY = "liquidity_sweep_fvg"  # e.g. 'liquidity_sweep_fvg', 'structure_continuation', 'ema_trend'

# Walk-Forward Settings
N_FOLDS = 5
IS_ANCHORED = True

# Optuna Settings
OPTUNA_STUDY_NAME = "study_liquidity_fvg_v1"
OPTUNA_N_TRIALS = 15

In [ ]:
# =============================================================================
# 2. ENVIRONMENT INITIALIZATION
# =============================================================================
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "src"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import polars as pl
from quant_platform.config.settings import settings
from quant_platform.data.storage.canonical import CanonicalStorage
from quant_platform.data.timeframes.resampler import CausalResampler
from quant_platform.strategies.catalog import StrategyCatalog
from quant_platform.research.walk_forward import WalkForwardEngine
from quant_platform.research.robustness import RobustnessEngine
from quant_platform.research.ablation import AblationEngine
from quant_platform.optimization.optuna_optimizer import OptunaOptimizer

settings.ensure_directories()
storage = CanonicalStorage()
df_1m = storage.read_range(symbol=SYMBOL)
df_tf = CausalResampler.resample(df_1m, target_timeframe=TIMEFRAME)
print(f"Loaded {SYMBOL} {TIMEFRAME} Dataset: {len(df_tf):,} bars")

strat_cls = StrategyCatalog.get_strategy_class(STRATEGY)
strategy = strat_cls()

In [ ]:
# =============================================================================
# 3. EXECUTE SELECTED VALIDATION OR OPTIMIZATION SUITE
# =============================================================================
if MODE == "WALK_FORWARD":
    wf_engine = WalkForwardEngine()
    report = wf_engine.run_walk_forward(
        df=df_tf,
        strategy=strategy,
        n_folds=N_FOLDS,
        is_anchored=IS_ANCHORED,
    )
    print(f"--- WALK-FORWARD EVALUATION REPORT ({report.mode}) ---")
    print(f"Strategy: {report.strategy_id}")
    print(f"Total Folds: {report.total_folds} | Profitable Folds: {report.profitable_folds} ({report.profitable_fold_ratio:.1f}%)")
    print(f"Aggregate OOS Return: {report.aggregate_oos_return:+.2f}%")
    print(f"Worst Fold Max Drawdown: {report.worst_fold_drawdown:.2f}%")
    print("\nFold Breakdown:")
    for f in report.folds:
        m = f.test_metrics
        print(f" - Fold {f.fold_index}: Train={f.train_bars} bars, Test={f.test_bars} bars | Trades={m.trade_count}, Net Ret={m.total_net_return:+.2f}%, MaxDD={m.max_drawdown_pct:.2f}%")

elif MODE == "ROBUSTNESS":
    rob_engine = RobustnessEngine(initial_capital=10000.0)
    report = rob_engine.run_robustness_suite(df_tf, strategy)
    print(report.summary)

elif MODE == "ABLATION":
    from quant_platform.strategies.advanced.liquidity_sweep_reversal import LiquiditySweepReversalStrategy
    from quant_platform.strategies.baselines.ema_trend import EmaTrendStrategy
    
    abl_engine = AblationEngine()
    variants = {
        "minus_fvg": LiquiditySweepReversalStrategy(),
        "baseline_ema": EmaTrendStrategy(),
    }
    res = abl_engine.run_ablation_study(df_tf, base_strategy=strategy, ablation_variants=variants)
    print(res.summary)

elif MODE == "OPTUNA":
    optimizer = OptunaOptimizer(study_name=OPTUNA_STUDY_NAME)
    
    def factory(params):
        return strat_cls(**params)
        
    def param_space(trial):
        return {
            "atr_period": trial.suggest_int("atr_period", 10, 20),
            "risk_reward_ratio": trial.suggest_float("risk_reward_ratio", 1.5, 3.0, step=0.5),
        }
        
    opt_res = optimizer.optimize_strategy(
        df=df_tf,
        strategy_factory=factory,
        param_space=param_space,
        n_trials=OPTUNA_N_TRIALS,
    )
    print(f"--- OPTUNA OPTIMIZATION CANDIDATE RESULT ---")
    print(f"Study: {opt_res.study_name} | Best Trial #{opt_res.best_trial_number}")
    print(f"Best Parameters: {opt_res.best_params}")
    print(f"Best Objective Value: {opt_res.best_value:.2f}")
    print(f"Verdict: {opt_res.verdict}")